# Python training - week 1

This is the first (maybe of many?) installments of our python problem setting. The aim of these tasks is to allow CCC colleagues to put their python skills to the test. We will hold a dedicated session for each problem to run through solutions and talk through difficulties that people may have encountered.

#### Task

Write a workflow which does the following:

- Reads in the 'total_generation.csv' and 'renewable_generation_by_source.csv' data from ./data/inputs/

- Calculates the renewable share of total generation across the full time period.

- Finds the first year in which renewable generation exceeded 40% of total generation.

- Finds the compound annual growth rate in renewable share of generation between 2008 and 2025.

- Uses this to extrapolate out what the proportion of renewable generation would be expected to be on current trends in 2030.

- Saves a csv in the ./data/outputs folder which contains the total generation, total renewable generation, and proportion of generation from renewable shares for 2000 - 2025.

- Bonus: 
    * Extrapolates out the full timeseries to 2030 based on the calculated compound annual growth rate.
    * Makes a plot of the historic and extrapolated data, showing the two as separate lines. Use ```matplotlib.pyplot``` for this.

#### Example solution below

In [ ]:
! pip install pandas
! pip install numpy

import pandas as pd
import numpy as np

In [ ]:
# reading in the two dataframes

total_df = pd.read_csv("./data/inputs/total_generation.csv")
renewable_df = pd.read_csv("./data/inputs/renewable_generation_by_source.csv")

In [ ]:
# inspecting each one in turn

total_df.head()

In [ ]:
renewable_df.head()

In [ ]:
# setting the "date" column as index in both to make joining easier later

total_df = total_df.set_index("date")
renewable_df = renewable_df.set_index("date")

In [ ]:
# getting total renewable generation as a new column

renewable_df["total_renewable_generation_twh"] = renewable_df.sum(axis=1)

renewable_df.tail()

In [ ]:
# joining the two dataframes together - note we don't actually have to do this 

combined_df = total_df.join(renewable_df["total_renewable_generation_twh"])

combined_df.tail()

In [ ]:
# calculating the share of generation as a new column

combined_df["renewable_proportion_of_gen"] = combined_df["total_renewable_generation_twh"] / combined_df["total_generation_twh"]

combined_df.tail()

In [ ]:
# finding the earliest date for which renewables exceeded 40%

exceeded_forty_perecent = combined_df["renewable_proportion_of_gen"] > 0.4

all_years = combined_df.loc[exceeded_forty_perecent]
first_year = all_years.index[0]

print(f"The first year in which renewables exceeded 40% of generation was {first_year}")

In [ ]:
# calculating the CAGR using the relevant formula

start_year = 2008
end_year = 2025
years_elapsed = end_year - start_year 

starting_value = combined_df["renewable_proportion_of_gen"][start_year]
end_value = combined_df["renewable_proportion_of_gen"][end_year]

cagr = ((end_value / starting_value) ** (1/years_elapsed) - 1)

In [ ]:
# extrapolating out to 2030 with this value
years_to_2030 = 2030 - 2025

projected_value = end_value * ((cagr + 1)**years_to_2030)

projected_value

In [ ]:
# exporting the data

combined_df.to_csv("./data/outputs/combined_generation_output.csv")

### Bonus answers below

In [ ]:
# Extrapolating the full timeseries

years_to_2030 = np.arange(2025, 2031, 1, dtype=int)
extrapolated_values = []

for year in years_to_2030:
    year_gap = year - end_year
    extrapolated_value = end_value * ((cagr + 1) ** year_gap)
    extrapolated_values.append(extrapolated_value)

extrapolated_values = np.array(extrapolated_values)

In [ ]:
# turning this into its own dataframe

extrapolated_2d_array = np.column_stack([years_to_2030, extrapolated_values])
extrapolated_df = pd.DataFrame(extrapolated_2d_array,
                               columns=["date", "extrapolated_renewable_share_of_generation"])

extrapolated_df = extrapolated_df.set_index("date")

In [ ]:
# concatenating the two dataframes to get one big dataframe

final_df = pd.concat([combined_df, extrapolated_df])

final_df.head()

In [ ]:
# and plotting the two together
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.append(PROJECT_ROOT)

import matplotlib.pyplot as plt
from utils import theme

fig, axs = plt.subplots()

axs.plot(final_df["renewable_proportion_of_gen"])
axs.plot(final_df["extrapolated_renewable_share_of_generation"])

axs.set_ylabel("Renewable proportion of total generation")
axs.legend(["Historic", "Extrapolation"])